# 🧠 Tipos de Memoria en LangChain
## Cómo los modelos de lenguaje recuerdan (y olvidan)

---

### ¿Qué aprenderás en esta actividad?

Los modelos de lenguaje, por diseño, **no tienen memoria entre llamadas**. Cada vez que envías un mensaje, el modelo no recuerda nada de lo anterior. LangChain resuelve esto con distintos tipos de memoria que gestionan el historial de conversación de maneras diferentes.

En esta actividad:
- Probarás **4 tipos de memoria** de forma aislada para ver exactamente qué guarda cada uno
- Construirás un **chat interactivo con Gradio** donde podrás cambiar el tipo de memoria en vivo
- Verás el **estado interno** de la memoria en tiempo real mientras chateas

---

### Estructura de la actividad

```
PARTE 1 ──► Instalación del entorno
PARTE 2 ──► Configuración de Ollama y modelo
PARTE 3 ──► ¿Qué es la memoria en LangChain?
PARTE 4 ──► ConversationBufferMemory
PARTE 5 ──► ConversationBufferWindowMemory
PARTE 6 ──► ConversationSummaryMemory
PARTE 7 ──► ConversationTokenBufferMemory
PARTE 8 ──► Chat interactivo con Gradio
PARTE 9 ──► Reflexión
PARTE 10 ─► Retos opcionales
PARTE 11 ─► [MÓDULO AVANZADO] EntityMemory y KGMemory
PARTE 12 ─► Publicación con Ngrok
```

> **Instrucción clave:** Ejecuta las celdas **de arriba hacia abajo**, una por una. No te saltes ninguna.

---
## PARTE 1 — Instalación y configuración del entorno

Instalamos todas las librerías necesarias para la actividad completa.

| Librería | Para qué sirve |
|---|---|
| `langchain` | Framework principal de orquestación de LLMs |
| `langchain-community` | Conectores con herramientas externas (Ollama, vectorstores) |
| `langchain-ollama` | Integración oficial de Ollama con LangChain |
| `gradio` | Interfaz web interactiva |
| `pyngrok` | Publicación del servidor con Ngrok |

In [ ]:
# ============================================================
# CELDA 1 — Instalación de dependencias
# ============================================================

# Instalamos todo en una sola celda de pip para ahorrar tiempo.
# El flag -q suprime la salida verbosa de pip.
# langchain-community incluye los conectores con Ollama y vectorstores.
!pip install -q langchain langchain-community langchain-ollama gradio pyngrok

print('✅ Librerías instaladas correctamente')

---
## PARTE 2 — Configuración de Ollama y selección de modelo

Ollama es un servidor local de modelos de lenguaje. Corre en segundo plano dentro de Colab y expone una API en `localhost:11434`.

```
  Tu código Python
       │
       ▼  petición HTTP
  Servidor Ollama  ← corre en localhost:11434
       │
       ▼  genera texto
    Modelo LLM
       │
       ▼  respuesta JSON
  Tu código Python
```

### ⚠️ Nota sobre el modelo
Empezaremos con **TinyLlama** porque es rápido y cabe en cualquier Colab gratuito. Sin embargo, cuando lleguemos a `ConversationSummaryMemory` (Parte 6), verás que los resúmenes pueden ser pobres. Ese es el momento ideal para cambiar a un modelo más potente.

In [ ]:
# ============================================================
# CELDA 2 — Instalación de Ollama en el sistema
# ============================================================

# Descargamos e instalamos el binario de Ollama directamente desde su script oficial.
# Este script detecta el sistema operativo y arquitectura automáticamente.
!curl -fsSL https://ollama.com/install.sh | sh

print('✅ Ollama instalado en el sistema')

In [ ]:
# ============================================================
# CELDA 3 — Lanzar el servidor Ollama en segundo plano
# ============================================================

import subprocess  # Para lanzar procesos del sistema desde Python
import time        # Para hacer pausas y esperar que el servidor inicie

# subprocess.Popen lanza el proceso en paralelo (no bloquea el notebook).
# Si usáramos subprocess.run(), el notebook quedaría congelado esperando
# que Ollama terminara (nunca termina, es un servidor).
# DEVNULL descarta la salida del servidor para no saturar el log del notebook.
ollama_process = subprocess.Popen(
    ['ollama', 'serve'],           # Equivalente a escribir 'ollama serve' en terminal
    stdout=subprocess.DEVNULL,     # Descarta la salida estándar del proceso
    stderr=subprocess.DEVNULL      # Descarta los mensajes de error del proceso
)

# Esperamos 3 segundos para que el servidor termine de iniciar
# antes de intentar hacer peticiones.
time.sleep(3)

print('✅ Servidor Ollama corriendo en localhost:11434')

In [ ]:
# ============================================================
# CELDA 4 — Descarga del modelo TinyLlama
# ============================================================

# TinyLlama es un modelo de ~600MB, ideal para Colab gratuito.
# 'ollama pull' descarga el modelo y lo deja listo para usar.
# Esta celda puede tardar 1-2 minutos dependiendo de la conexión.
!ollama pull tinyllama

print('✅ Modelo TinyLlama descargado y listo')

In [ ]:
# ============================================================
# CELDA 5 — Configurar el modelo activo
# ============================================================

# Esta variable define qué modelo usará toda la actividad.
# Más adelante te sugeriremos cambiarla a 'llama3.2' para mejores resultados.
MODELO_ACTIVO = 'tinyllama'

# Si ya tienes llama3.2 descargado y quieres mejores respuestas, cambia a:
# MODELO_ACTIVO = 'llama3.2'
# Y ejecuta: !ollama pull llama3.2

print(f'✅ Modelo configurado: {MODELO_ACTIVO}')

In [ ]:
# ============================================================
# CELDA 6 — Prueba rápida del modelo
# ============================================================

from langchain_ollama import ChatOllama  # Conector oficial de Ollama para LangChain
from langchain_core.messages import HumanMessage  # Representa un mensaje del usuario

# Creamos una instancia del modelo.
# temperature=0.7 controla la creatividad: 0 = respuestas deterministas, 1 = muy creativo.
llm = ChatOllama(
    model=MODELO_ACTIVO,
    temperature=0.7   # Balance entre coherencia y creatividad
)

# Enviamos un mensaje simple para verificar que el modelo responde.
# HumanMessage envuelve el texto como si fuera un turno del usuario en una conversación.
respuesta = llm.invoke([HumanMessage(content='Responde en una sola oración: ¿qué es la memoria en los chatbots?')])

print('Respuesta del modelo:')
print(respuesta.content)
print()
print(f'✅ Modelo {MODELO_ACTIVO} funcionando correctamente')

---
## PARTE 3 — ¿Qué es la memoria en LangChain?

### El problema fundamental

Cuando llamas a un LLM, este solo conoce lo que está en el **contexto actual**. No recuerda nada de llamadas anteriores. Si quieres que el modelo "recuerde" la conversación, necesitas enviarle el historial tú mismo en cada llamada.

```
Sin memoria:
  Turno 1: Usuario: "Me llamo Ana"  → Modelo: "Hola Ana!"
  Turno 2: Usuario: "¿Cómo me llamo?" → Modelo: "No lo sé"

Con memoria:
  Turno 1: Usuario: "Me llamo Ana"  → Modelo: "Hola Ana!"
  Turno 2: [historial + "¿Cómo me llamo?"] → Modelo: "Te llamas Ana"
```

### Los 4 tipos de memoria que exploraremos

| Tipo | ¿Qué guarda? | Ventaja | Desventaja |
|---|---|---|---|
| `BufferMemory` | Todo el historial completo | No pierde nada | Crece ilimitado |
| `WindowMemory` | Últimas N interacciones | Tamaño controlado | Olvida lo antiguo |
| `SummaryMemory` | Resumen progresivo | Compacta el historial | Requiere un buen modelo |
| `TokenBufferMemory` | Historial hasta X tokens | Control preciso | Corta en medio de ideas |

### Arquitectura de LangChain con memoria

```
  Usuario escribe
       │
       ▼
  [Objeto Memoria]  ← guarda y recupera el historial
       │
       ▼  historial + mensaje nuevo
  [LLM - Ollama]    ← recibe contexto completo
       │
       ▼  respuesta
  [Objeto Memoria]  ← actualiza el historial con la respuesta
       │
       ▼
  Usuario ve respuesta
```

---
## PARTE 4 — ConversationBufferMemory

### ¿Qué hace?
Guarda **todo el historial** de la conversación, sin límite. Cada turno se agrega al buffer. El modelo recibe siempre la conversación completa.

```
Turno 1: [H: hola] [AI: hola!]
Turno 2: [H: hola] [AI: hola!] [H: ¿qué es Python?] [AI: un lenguaje...]
Turno 3: [H: hola] [AI: hola!] [H: ¿qué es Python?] [AI: un lenguaje...] [H: dame un ejemplo] [AI: ...]
         ↑ El buffer crece con cada turno
```

**Cuándo usarla:** Conversaciones cortas donde necesitas contexto completo.

In [ ]:
# ============================================================
# CELDA 7 — Configurar ConversationBufferMemory
# ============================================================

from langchain.memory import ConversationBufferMemory          # Guarda todo el historial
from langchain.chains import ConversationChain                 # Cadena que conecta memoria + LLM

# Creamos el objeto de memoria.
# return_messages=True devuelve el historial como lista de objetos HumanMessage/AIMessage.
# Esto es necesario para que funcione con modelos de chat (vs modelos de completado de texto).
memoria_buffer = ConversationBufferMemory(
    return_messages=True   # Formato de mensajes estructurados, no texto plano
)

# ConversationChain conecta el LLM con la memoria automáticamente.
# verbose=False suprime el log detallado de cada llamada al LLM.
cadena_buffer = ConversationChain(
    llm=llm,
    memory=memoria_buffer,
    verbose=False
)

print('✅ ConversationBufferMemory configurada')

In [ ]:
# ============================================================
# CELDA 8 — Probar BufferMemory: enviar mensajes y ver el estado
# ============================================================

# Función auxiliar para enviar un mensaje e imprimir el estado interno de la memoria.
# Esto nos permite VER exactamente qué está guardando la memoria en cada turno.
def chatear_y_mostrar(cadena, memoria, mensaje):
    print(f'👤 Usuario: {mensaje}')
    
    # cadena.predict() envía el mensaje al LLM con el historial de memoria incluido.
    respuesta = cadena.predict(input=mensaje)
    print(f'🤖 Modelo:  {respuesta}')
    
    # Mostramos el estado interno de la memoria después de cada turno.
    # memoria.chat_memory.messages contiene la lista completa de mensajes.
    mensajes = memoria.chat_memory.messages
    print(f'\n📦 Estado de la memoria ({len(mensajes)} mensajes guardados):')
    for i, msg in enumerate(mensajes):
        tipo = '👤 Human' if msg.type == 'human' else '🤖 AI'
        # Truncamos a 60 chars para que sea legible en consola
        contenido_corto = msg.content[:60] + '...' if len(msg.content) > 60 else msg.content
        print(f'  [{i+1}] {tipo}: {contenido_corto}')
    print('-' * 60)

print('✅ Función de demostración lista. Iniciando conversación de prueba...')
print('=' * 60)

In [ ]:
# ============================================================
# CELDA 9 — Conversación de prueba con BufferMemory
# ============================================================

# Observa cómo la memoria crece con cada turno.
# El modelo recuerda el nombre porque todo el historial se envía siempre.

chatear_y_mostrar(cadena_buffer, memoria_buffer, 'Hola, me llamo Carlos y estudio ingeniería.')
chatear_y_mostrar(cadena_buffer, memoria_buffer, '¿Cuál es mi nombre?')
chatear_y_mostrar(cadena_buffer, memoria_buffer, '¿Qué estudio?')

print('✅ Prueba de BufferMemory completada')
print('   Observa cómo el buffer crece: 2 → 4 → 6 mensajes')

---
## PARTE 5 — ConversationBufferWindowMemory

### ¿Qué hace?
Guarda solo las **últimas K interacciones** (donde una interacción = 1 mensaje de usuario + 1 respuesta). Lo más antiguo se elimina automáticamente.

```
Con k=2 (guarda las últimas 2 interacciones):

Turno 1: [H1][AI1]
Turno 2: [H1][AI1] [H2][AI2]
Turno 3:           [H2][AI2] [H3][AI3]  ← H1 y AI1 se eliminan
Turno 4:                     [H3][AI3] [H4][AI4]  ← H2 y AI2 se eliminan
         ↑ La ventana desliza: siempre conserva las últimas 2
```

**Cuándo usarla:** Cuando el historial puede volverse muy largo pero el contexto reciente es suficiente.

In [ ]:
# ============================================================
# CELDA 10 — Configurar ConversationBufferWindowMemory
# ============================================================

from langchain.memory import ConversationBufferWindowMemory   # Ventana deslizante de historial

# k=2 significa que guardamos las últimas 2 interacciones (2 pares Human/AI).
# Puedes cambiar k para experimentar con ventanas más grandes o más pequeñas.
memoria_window = ConversationBufferWindowMemory(
    k=2,                   # Número de interacciones a conservar
    return_messages=True   # Formato de objetos Message (necesario para chat models)
)

cadena_window = ConversationChain(
    llm=llm,
    memory=memoria_window,
    verbose=False
)

print('✅ ConversationBufferWindowMemory configurada con k=2')
print('   (Guardará solo las últimas 2 interacciones)')

In [ ]:
# ============================================================
# CELDA 11 — Conversación de prueba con WindowMemory
# ============================================================

# Observa lo que pasa en el turno 3:
# el modelo ya no recuerda la información del turno 1 porque la ventana la eliminó.

chatear_y_mostrar(cadena_window, memoria_window, 'Hola, me llamo Sofía.')
chatear_y_mostrar(cadena_window, memoria_window, 'Mi color favorito es el azul.')
chatear_y_mostrar(cadena_window, memoria_window, '¿Cómo me llamo?')

# En este punto, la ventana solo tiene los turnos 2 y 3.
# El nombre del turno 1 ya no está en la memoria.
print()
print('⚠️  Nota: Si el modelo respondió que no sabe tu nombre, es correcto.')
print('   El turno 1 ("me llamo Sofía") fue eliminado de la ventana.')
print('✅ Prueba de WindowMemory completada')

---
## PARTE 6 — ConversationSummaryMemory

### ¿Qué hace?
En lugar de guardar los mensajes exactos, usa el LLM para **resumir progresivamente** la conversación. El resumen se actualiza con cada turno.

```
Turno 1: [Resumen: "El usuario saludó"]
Turno 2: [Resumen: "El usuario se llama Ana y estudia física"]
Turno 3: [Resumen: "Ana estudia física y preguntó sobre mecánica cuántica"]
         ↑ Siempre es un solo bloque de texto, no una lista de mensajes
```

**Cuándo usarla:** Conversaciones muy largas donde los detalles exactos no importan.

---

### 💡 Momento ideal para cambiar de modelo

Esta memoria **usa el LLM para resumir**. Con TinyLlama, los resúmenes pueden ser imprecisos o muy cortos. Si quieres ver esta memoria funcionando bien, este es el momento de cambiar a `llama3.2`:

```python
# En la Celda 5, cambia a:
MODELO_ACTIVO = 'llama3.2'
# Y ejecuta: !ollama pull llama3.2
# Luego vuelve a ejecutar la Celda 6 para recrear el objeto llm.
```

Si prefieres continuar con TinyLlama está bien, pero toma los resúmenes con cautela.

In [ ]:
# ============================================================
# CELDA 12 — Configurar ConversationSummaryMemory
# ============================================================

from langchain.memory import ConversationSummaryMemory   # Memoria que resume el historial

# Esta memoria necesita un LLM para hacer los resúmenes.
# Usamos el mismo llm que configuramos antes.
# Cada vez que se guarda un nuevo mensaje, el LLM genera un nuevo resumen.
memoria_summary = ConversationSummaryMemory(
    llm=llm,               # El LLM que se usará para resumir
    return_messages=True   # Devuelve el resumen como mensaje SystemMessage
)

cadena_summary = ConversationChain(
    llm=llm,
    memory=memoria_summary,
    verbose=False
)

print('✅ ConversationSummaryMemory configurada')
print(f'   Usando {MODELO_ACTIVO} para generar resúmenes')

In [ ]:
# ============================================================
# CELDA 13 — Función de demostración adaptada para SummaryMemory
# ============================================================

# SummaryMemory guarda un resumen en texto, no una lista de mensajes.
# Necesitamos una función diferente para mostrar su estado interno.
def chatear_y_mostrar_summary(cadena, memoria, mensaje):
    print(f'👤 Usuario: {mensaje}')
    respuesta = cadena.predict(input=mensaje)
    print(f'🤖 Modelo:  {respuesta}')
    
    # En SummaryMemory, el estado se guarda como un string de resumen,
    # no como una lista de mensajes. Accedemos a él así:
    resumen = memoria.moving_summary_buffer  # El resumen acumulado hasta este momento
    print(f'\n📦 Resumen actual en memoria:')
    if resumen:
        print(f'  "{resumen}"')
    else:
        print('  (vacío - aún no hay suficiente conversación para resumir)')
    print('-' * 60)

print('✅ Función de demostración para SummaryMemory lista')

In [ ]:
# ============================================================
# CELDA 14 — Conversación de prueba con SummaryMemory
# ============================================================

# Observa cómo el resumen se actualiza con cada turno.
# No se guardan los mensajes exactos, sino una síntesis comprimida.

chatear_y_mostrar_summary(cadena_summary, memoria_summary, 'Hola, soy Juan y trabajo como diseñador gráfico.')
chatear_y_mostrar_summary(cadena_summary, memoria_summary, 'Estoy aprendiendo sobre inteligencia artificial.')
chatear_y_mostrar_summary(cadena_summary, memoria_summary, '¿Qué recuerdas de mí?')

print('✅ Prueba de SummaryMemory completada')
print('   El modelo no guardó los mensajes exactos, sino un resumen.')

---
## PARTE 7 — ConversationTokenBufferMemory

### ¿Qué hace?
Guarda el historial completo hasta un **límite máximo de tokens**. Cuando se supera el límite, elimina los mensajes más antiguos.

```
Con max_token_limit=100:

Turno 1: [H1:15tok][AI1:20tok]  → Total: 35 tokens  ✅
Turno 2: [H1:15][AI1:20][H2:18][AI2:25] → Total: 78 tokens  ✅
Turno 3: [H1:15][AI1:20][H2:18][AI2:25][H3:15][AI3:30] → 123 tokens  ❌
         Elimina H1+AI1:  [H2:18][AI2:25][H3:15][AI3:30] → 88 tokens  ✅
```

**Cuándo usarla:** Cuando el costo o límite del modelo se mide en tokens y necesitas control preciso.

In [ ]:
# ============================================================
# CELDA 15 — Configurar ConversationTokenBufferMemory
# ============================================================

from langchain.memory import ConversationTokenBufferMemory   # Memoria con límite en tokens

# max_token_limit=200 es muy bajo a propósito para ver el efecto rápidamente.
# En producción usarías valores como 2000 o 4000 según el modelo.
# Esta memoria también necesita el LLM para contar tokens correctamente.
memoria_token = ConversationTokenBufferMemory(
    llm=llm,                   # Necesario para contar tokens según el modelo
    max_token_limit=200,       # Máximo de tokens a conservar en memoria
    return_messages=True       # Formato de objetos Message
)

cadena_token = ConversationChain(
    llm=llm,
    memory=memoria_token,
    verbose=False
)

print('✅ ConversationTokenBufferMemory configurada')
print('   Límite: 200 tokens (bajo a propósito para ver el efecto)')

In [ ]:
# ============================================================
# CELDA 16 — Conversación de prueba con TokenBufferMemory
# ============================================================

# Enviamos mensajes relativamente largos para llegar rápido al límite de tokens.

chatear_y_mostrar(cadena_token, memoria_token, 'Hola, me llamo Roberto y soy doctor especialista en cardiología.')
chatear_y_mostrar(cadena_token, memoria_token, 'Trabajo en el hospital central de la ciudad desde hace diez años.')
chatear_y_mostrar(cadena_token, memoria_token, 'También tengo un consultorio privado los fines de semana.')
chatear_y_mostrar(cadena_token, memoria_token, '¿Cuál es mi nombre y profesión?')

print('✅ Prueba de TokenBufferMemory completada')
print('   Observa si los primeros mensajes desaparecieron al llegar al límite de tokens.')

---
## PARTE 8 — Chat Interactivo con Gradio

### Lo que construiremos

Una interfaz de chat donde el estudiante puede:
1. Elegir el tipo de memoria desde un menú desplegable
2. Chatear con el modelo normalmente
3. Ver en tiempo real el **estado interno** de la memoria en un panel lateral
4. Cambiar de memoria en cualquier momento (el chat se reinicia con aviso)

```
┌─────────────────────────────────────────────────────┐
│  Tipo de memoria: [Dropdown ▼]                      │
├───────────────────────────┬─────────────────────────┤
│                           │                         │
│   Chat                    │  Estado de la memoria   │
│   ─────                   │  ─────────────────────  │
│   [historial de mensajes] │  [contenido interno]    │
│                           │                         │
│   [Escribe tu mensaje...] │                         │
│   [Enviar] [Limpiar]      │                         │
└───────────────────────────┴─────────────────────────┘
```

In [ ]:
# ============================================================
# CELDA 17 — Importaciones y configuración de la interfaz
# ============================================================

import gradio as gr                                           # Framework para la interfaz web
from langchain.memory import (                               # Los 4 tipos de memoria
    ConversationBufferMemory,
    ConversationBufferWindowMemory,
    ConversationSummaryMemory,
    ConversationTokenBufferMemory
)
from langchain.chains import ConversationChain               # Cadena memoria + LLM
from langchain_ollama import ChatOllama                      # Conector con Ollama

# Recreamos el LLM para asegurarnos de usar la configuración actual.
llm_gradio = ChatOllama(
    model=MODELO_ACTIVO,
    temperature=0.7
)

# Diccionario que mapea el nombre legible a su parámetro de creación.
# Usaremos este mapa para crear la memoria correcta según la selección del usuario.
TIPOS_MEMORIA = {
    '📚 Buffer (historial completo)': 'buffer',
    '🪟 Window (últimas 3 interacciones)': 'window',
    '📝 Summary (resumen progresivo)': 'summary',
    '🔢 Token Buffer (límite 300 tokens)': 'token'
}

print('✅ Importaciones y configuración listas')

In [ ]:
# ============================================================
# CELDA 18 — Función para crear la memoria según el tipo elegido
# ============================================================

def crear_memoria(tipo):
    """
    Crea y devuelve el objeto de memoria correcto según el tipo solicitado.
    También devuelve una cadena ConversationChain lista para usar.
    """
    if tipo == 'buffer':
        # Guarda todo el historial sin límite
        memoria = ConversationBufferMemory(return_messages=True)

    elif tipo == 'window':
        # Guarda solo las últimas 3 interacciones (k=3)
        memoria = ConversationBufferWindowMemory(
            k=3,                   # Ventana de 3 interacciones
            return_messages=True
        )

    elif tipo == 'summary':
        # Usa el LLM para resumir el historial progresivamente
        memoria = ConversationSummaryMemory(
            llm=llm_gradio,        # El LLM que generará los resúmenes
            return_messages=True
        )

    elif tipo == 'token':
        # Guarda historial hasta 300 tokens
        memoria = ConversationTokenBufferMemory(
            llm=llm_gradio,        # Necesario para contar tokens
            max_token_limit=300,   # Límite de tokens en el historial
            return_messages=True
        )

    # Creamos la cadena que conecta la memoria con el LLM
    cadena = ConversationChain(
        llm=llm_gradio,
        memory=memoria,
        verbose=False
    )

    # Devolvemos ambos objetos para poder acceder al estado interno de la memoria
    return memoria, cadena

print('✅ Función crear_memoria() definida')

In [ ]:
# ============================================================
# CELDA 19 — Función para leer el estado interno de la memoria
# ============================================================

def obtener_estado_memoria(memoria, tipo):
    """
    Lee el estado interno de la memoria y lo formatea para mostrar en el panel lateral.
    Cada tipo de memoria guarda el estado de forma diferente.
    """
    if tipo == 'summary':
        # SummaryMemory guarda un string de resumen, no una lista de mensajes.
        resumen = memoria.moving_summary_buffer
        if resumen:
            return f'📝 **Resumen acumulado:**\n\n{resumen}'
        else:
            return '📝 Resumen: (aún vacío, sigue chateando)'
    else:
        # Buffer, Window y Token guardan listas de mensajes.
        mensajes = memoria.chat_memory.messages
        if not mensajes:
            return '(Memoria vacía — escribe tu primer mensaje)'

        # Formateamos cada mensaje para mostrarlo claramente
        lineas = [f'💾 **{len(mensajes)} mensajes en memoria:**\n']
        for i, msg in enumerate(mensajes):
            # msg.type es 'human' o 'ai' según quién envió el mensaje
            icono = '👤' if msg.type == 'human' else '🤖'
            # Truncamos a 80 chars para que quepa en el panel
            contenido = msg.content[:80] + '...' if len(msg.content) > 80 else msg.content
            lineas.append(f'{icono} **[{i+1}]** {contenido}')

        return '\n\n'.join(lineas)

print('✅ Función obtener_estado_memoria() definida')

In [ ]:
# ============================================================
# CELDA 20 — Estado global de la aplicación
# ============================================================

# Usamos un diccionario mutable como estado global.
# Gradio ejecuta las funciones en múltiples hilos, por eso necesitamos
# un objeto mutable compartido en lugar de variables simples.
estado_app = {
    'tipo': 'buffer',           # Tipo de memoria activo por defecto
    'memoria': None,            # Objeto de memoria actual
    'cadena': None              # Cadena LangChain actual
}

# Inicializamos con la memoria por defecto (buffer)
estado_app['memoria'], estado_app['cadena'] = crear_memoria('buffer')

print('✅ Estado inicial de la aplicación configurado')
print(f'   Memoria activa: {estado_app["tipo"]}')

In [ ]:
# ============================================================
# CELDA 21 — Funciones de lógica del chat para Gradio
# ============================================================

def cambiar_memoria(tipo_legible, historial):
    """
    Se ejecuta cuando el usuario cambia el tipo de memoria en el dropdown.
    Reinicia el chat y crea una nueva memoria del tipo seleccionado.
    """
    # Convertimos el nombre legible ('📚 Buffer...') al código interno ('buffer')
    tipo_codigo = TIPOS_MEMORIA[tipo_legible]

    # Actualizamos el estado global con la nueva memoria
    estado_app['tipo'] = tipo_codigo
    estado_app['memoria'], estado_app['cadena'] = crear_memoria(tipo_codigo)

    # Añadimos un mensaje del sistema al historial de Gradio informando el cambio
    # None como primer elemento indica que no hay input del usuario en este turno
    mensaje_aviso = f'⚠️ Memoria cambiada a **{tipo_legible}**. El historial fue reiniciado.'
    historial_nuevo = historial + [[None, mensaje_aviso]]

    # Actualizamos el panel lateral con el estado de la nueva memoria (vacía)
    estado_panel = obtener_estado_memoria(estado_app['memoria'], tipo_codigo)

    return historial_nuevo, estado_panel


def responder(mensaje_usuario, historial):
    """
    Se ejecuta cuando el usuario envía un mensaje.
    Obtiene la respuesta del LLM con memoria y actualiza el panel de estado.
    """
    if not mensaje_usuario.strip():
        # Si el mensaje está vacío, no hacemos nada
        return historial, obtener_estado_memoria(estado_app['memoria'], estado_app['tipo']), ''

    # Enviamos el mensaje al LLM a través de la cadena con memoria
    # .predict() agrega el mensaje a la memoria y obtiene la respuesta del LLM
    try:
        respuesta = estado_app['cadena'].predict(input=mensaje_usuario)
    except Exception as e:
        respuesta = f'Error al consultar el modelo: {str(e)}'

    # Agregamos el par (usuario, modelo) al historial de Gradio
    historial_nuevo = historial + [[mensaje_usuario, respuesta]]

    # Leemos el estado actualizado de la memoria para el panel lateral
    estado_panel = obtener_estado_memoria(estado_app['memoria'], estado_app['tipo'])

    # Devolvemos: historial actualizado, estado de memoria, campo de texto vacío
    return historial_nuevo, estado_panel, ''


def limpiar_chat():
    """
    Reinicia el chat manteniendo el tipo de memoria activo.
    """
    estado_app['memoria'], estado_app['cadena'] = crear_memoria(estado_app['tipo'])
    return [], '(Memoria reiniciada — escribe tu primer mensaje)', ''

print('✅ Funciones de lógica del chat definidas')

In [ ]:
# ============================================================
# CELDA 22 — Construcción de la interfaz Gradio
# ============================================================

# CSS personalizado para dar una apariencia más polida a la interfaz.
# Gradio permite inyectar estilos CSS para personalizar componentes específicos.
CSS = """
#panel-memoria {
    background: #1e1e2e;
    border: 1px solid #45475a;
    border-radius: 8px;
    padding: 16px;
    font-family: monospace;
    font-size: 13px;
    color: #cdd6f4;
    min-height: 400px;
    overflow-y: auto;
}
#panel-memoria h3 {
    color: #89b4fa;
    margin-top: 0;
}
.tipo-badge {
    background: #313244;
    border-radius: 4px;
    padding: 2px 8px;
    color: #a6e3a1;
    font-size: 12px;
}
"""

# Construimos la interfaz usando gr.Blocks, que permite diseño personalizado
# a diferencia de gr.Interface que tiene un diseño fijo.
with gr.Blocks(css=CSS, title='Memoria en LangChain') as demo:

    # Título principal
    gr.Markdown("""
    # 🧠 Tipos de Memoria en LangChain
    Cambia el tipo de memoria y observa cómo cambia lo que el modelo "recuerda".
    El panel derecho muestra el **estado interno real** de la memoria.
    """)

    # Selector de tipo de memoria — ocupa toda la fila
    selector_memoria = gr.Dropdown(
        choices=list(TIPOS_MEMORIA.keys()),     # Opciones del dropdown
        value=list(TIPOS_MEMORIA.keys())[0],    # Valor inicial: Buffer
        label='📋 Tipo de memoria activa',
        info='Al cambiar, el historial del chat se reiniciará automáticamente'
    )

    # Layout de dos columnas: chat a la izquierda, estado de memoria a la derecha
    with gr.Row():

        # Columna izquierda: el chat
        with gr.Column(scale=3):  # scale=3 hace esta columna 3x más ancha
            chatbot = gr.Chatbot(
                label='💬 Chat',
                height=450,
                bubble_full_width=False     # Los mensajes no ocupan todo el ancho
            )

            # Fila con el campo de texto y el botón de enviar
            with gr.Row():
                campo_texto = gr.Textbox(
                    placeholder='Escribe un mensaje... (prueba diciendo tu nombre)',
                    label='',
                    scale=4,               # El campo de texto ocupa 4 partes
                    lines=1
                )
                btn_enviar = gr.Button('Enviar ▶', variant='primary', scale=1)

            btn_limpiar = gr.Button('🗑️ Limpiar chat', variant='secondary')

        # Columna derecha: estado interno de la memoria
        with gr.Column(scale=2):  # scale=2 hace esta columna 2/5 del ancho total
            panel_memoria = gr.Markdown(
                value='(Memoria vacía — escribe tu primer mensaje)',
                label='🔍 Estado interno de la memoria',
                elem_id='panel-memoria'
            )

    # ── Conexión de eventos ──────────────────────────────────

    # Cuando el usuario cambia el dropdown, llamamos a cambiar_memoria()
    selector_memoria.change(
        fn=cambiar_memoria,
        inputs=[selector_memoria, chatbot],     # Qué datos recibe la función
        outputs=[chatbot, panel_memoria]        # Qué componentes actualiza
    )

    # Cuando el usuario hace clic en Enviar, llamamos a responder()
    btn_enviar.click(
        fn=responder,
        inputs=[campo_texto, chatbot],
        outputs=[chatbot, panel_memoria, campo_texto]  # campo_texto se vacía
    )

    # También enviamos con Enter (submit del Textbox)
    campo_texto.submit(
        fn=responder,
        inputs=[campo_texto, chatbot],
        outputs=[chatbot, panel_memoria, campo_texto]
    )

    # El botón de limpiar reinicia el chat
    btn_limpiar.click(
        fn=limpiar_chat,
        inputs=[],
        outputs=[chatbot, panel_memoria, campo_texto]
    )

    # Sección de instrucciones para el estudiante
    gr.Markdown("""
    ---
    ### 💡 Cómo explorar los tipos de memoria
    1. **Empieza con Buffer:** dile al modelo tu nombre y pregúntaselo 5 mensajes después.
    2. **Cambia a Window (k=3):** continúa chateando hasta que pierda información anterior.
    3. **Prueba Summary:** cuenta una historia larga y observa cómo se comprime en el panel.
    4. **Prueba Token Buffer:** envía mensajes largos y observa cuándo empiezan a desaparecer.
    """)

print('✅ Interfaz Gradio construida')
print('   Ejecuta la siguiente celda para lanzarla')

In [ ]:
# ============================================================
# CELDA 23 — Lanzar la interfaz (prueba local)
# ============================================================

# Lanzamos en modo local para verificar que la interfaz funciona
# antes de publicarla con Ngrok.
# server_name='0.0.0.0' permite que la interfaz sea accesible desde Ngrok.
# share=False porque usaremos Ngrok para el enlace público (más estable).
demo.launch(
    server_name='0.0.0.0',   # Escucha en todas las interfaces de red
    server_port=7860,         # Puerto estándar de Gradio
    share=False               # No usamos el share de Gradio (usaremos Ngrok)
)

---
## PARTE 9 — Reflexión

Responde las siguientes preguntas en la celda de texto que aparece después. Puedes usar tus propias palabras, no hay respuestas incorrectas.

**Responde aquí:**

1. **Conceptual:** ¿Cuál es la diferencia fundamental entre `BufferMemory` y `WindowMemory`? ¿En qué situación real preferirías una sobre la otra?

   *Tu respuesta:*

---

2. **De código:** ¿Por qué `ConversationSummaryMemory` y `ConversationTokenBufferMemory` necesitan recibir un LLM como parámetro, mientras que `BufferMemory` y `WindowMemory` no?

   *Tu respuesta:*

---

3. **Implicaciones:** Si una empresa usa `ConversationSummaryMemory` para su chatbot de atención al cliente, ¿qué riesgos podría haber al usar el mismo LLM para resumir y para responder? Piensa en costos, errores de resumen y privacidad.

   *Tu respuesta:*

---

4. **Comparativa:** Compara `BufferMemory` vs `TokenBufferMemory`. ¿Cuándo elegirías `TokenBuffer` aunque sea más compleja de configurar?

   *Tu respuesta:*

---

5. **Reflexión personal:** Después de experimentar con el chat interactivo, ¿cuál tipo de memoria te pareció más útil para un asistente de aprendizaje? ¿Por qué?

   *Tu respuesta:*

---
## PARTE 10 — Retos Opcionales

Estos retos no tienen solución incluida. Son para que experimentes por tu cuenta.

**Reto 1 — Ajusta la ventana y observa:**
Cambia el parámetro `k` en `WindowMemory` a `k=1` y `k=5`. ¿Cuándo el modelo empieza a "olvidar" información importante? Pista: Busca cómo re-crear la memoria con diferentes valores de `k` y volver a añadirla al dropdown.

**Reto 2 — Combina memoria con personalidad:**
Agrega un System Prompt al modelo para que tenga una personalidad definida (ej: un asistente de cocina). ¿La personalidad se conserva correctamente con todos los tipos de memoria? Pista: Investiga `SystemMessage` en LangChain y cómo pasarlo a `ConversationChain`.

**Reto 3 — Mide el uso de tokens:**
Modifica la función `responder()` para que imprima cuántos tokens consume cada llamada al LLM. Observa cómo crece el consumo con `BufferMemory` vs `TokenBufferMemory`. Pista: Investiga el atributo `usage_metadata` de las respuestas de LangChain.

In [ ]:
# ============================================================
# CELDA VACÍA — Espacio para tus experimentos
# ============================================================

# Escribe aquí tu código para los retos opcionales


---
---
## PARTE 11 — 🚀 MÓDULO AVANZADO: EntityMemory y KGMemory

> **⚠️ Este módulo es opcional.** Requiere un modelo más capaz que TinyLlama. Se recomienda usar `llama3.2` o superior.

---

### ¿Por qué estas memorias son "avanzadas"?

Las 4 memorias anteriores tratan el historial como **texto lineal** (mensajes uno tras otro). Las memorias avanzadas extraen **estructura** del historial: qué entidades existen y cómo se relacionan entre sí.

| Tipo | ¿Qué extrae? | Estructura interna |
|---|---|---|
| `EntityMemory` | Personas, lugares, conceptos mencionados | Diccionario: `{"Ana": "doctora de 30 años"}` |
| `KGMemory` | Relaciones entre entidades | Grafo: `Ana → trabaja_en → Hospital` |

### Diagrama de EntityMemory

```
Conversación:                    Estado interno:
"Ana es doctora"           →     { "Ana": "es doctora" }
"Ana trabaja con Pedro"    →     { "Ana": "es doctora, trabaja con Pedro",
                                   "Pedro": "colega de Ana" }
"Pedro vive en Madrid"     →     { "Ana": "es doctora, trabaja con Pedro",
                                   "Pedro": "colega de Ana, vive en Madrid",
                                   "Madrid": "ciudad donde vive Pedro" }
```

In [ ]:
# ============================================================
# CELDA 24 — [AVANZADO] Cambiar a un modelo más potente
# ============================================================

# EntityMemory y KGMemory necesitan un modelo que pueda extraer entidades
# y relaciones del texto. TinyLlama frecuentemente falla en esta tarea.

# Descargamos llama3.2 si no lo tenemos aún.
# Este modelo pesa ~2GB, puede tardar varios minutos en Colab.
!ollama pull llama3.2

# Creamos una instancia del modelo más potente para el módulo avanzado.
# No reemplazamos llm_gradio para no afectar la interfaz principal.
llm_avanzado = ChatOllama(
    model='llama3.2',
    temperature=0.3     # Temperatura más baja para extracciones más precisas
)

print('✅ Modelo llama3.2 listo para el módulo avanzado')

### 11A — ConversationEntityMemory

Esta memoria identifica **entidades** (personas, lugares, objetos, conceptos) mencionadas en la conversación y mantiene un perfil actualizado de cada una. Cuando el usuario vuelve a mencionar una entidad, el modelo recibe su perfil completo.

In [ ]:
# ============================================================
# CELDA 25 — [AVANZADO] Configurar ConversationEntityMemory
# ============================================================

from langchain.memory import ConversationEntityMemory         # Memoria de entidades
from langchain.memory.prompt import ENTITY_MEMORY_CONVERSATION_TEMPLATE  # Prompt especial

# EntityMemory necesita un prompt especial que le indica al LLM
# cómo mantener y actualizar el registro de entidades.
# ENTITY_MEMORY_CONVERSATION_TEMPLATE es el prompt oficial de LangChain para esto.
memoria_entity = ConversationEntityMemory(
    llm=llm_avanzado   # El LLM que extrae y actualiza entidades del historial
)

# ConversationChain con el prompt especial para entidades.
# Este prompt incluye una sección para las entidades conocidas.
cadena_entity = ConversationChain(
    llm=llm_avanzado,
    memory=memoria_entity,
    prompt=ENTITY_MEMORY_CONVERSATION_TEMPLATE,   # Prompt que sabe manejar entidades
    verbose=False
)

print('✅ ConversationEntityMemory configurada con llama3.2')

In [ ]:
# ============================================================
# CELDA 26 — [AVANZADO] Función para mostrar estado de EntityMemory
# ============================================================

def mostrar_entidades(memoria):
    """
    EntityMemory guarda las entidades en un diccionario interno.
    Esta función lo imprime de forma legible.
    """
    # memoria.entity_store.store es el diccionario interno de entidades.
    # Cada clave es el nombre de la entidad, el valor es lo que el modelo sabe de ella.
    entidades = memoria.entity_store.store

    if not entidades:
        print('  (No se han detectado entidades aún)')
        return

    print(f'  📋 {len(entidades)} entidades detectadas:')
    for entidad, descripcion in entidades.items():
        print(f'  • "{entidad}" → {descripcion[:100]}...' if len(descripcion) > 100 else f'  • "{entidad}" → {descripcion}')

print('✅ Función mostrar_entidades() definida')

In [ ]:
# ============================================================
# CELDA 27 — [AVANZADO] Prueba de EntityMemory
# ============================================================

# Introducimos varias entidades en la conversación.
# Observa cómo el modelo construye perfiles de cada persona mencionada.

def chatear_entity(mensaje):
    print(f'👤 Usuario: {mensaje}')
    respuesta = cadena_entity.predict(input=mensaje)
    print(f'🤖 Modelo:  {respuesta}')
    print()
    print('📦 Estado de entidades:')
    mostrar_entidades(memoria_entity)
    print('-' * 60)

chatear_entity('Mi amiga Laura es ingeniera de software y trabaja en Google.')
chatear_entity('Laura y su colega Marco están desarrollando una IA para diagnóstico médico.')
chatear_entity('¿Qué sabes sobre Laura?')

print('✅ Prueba de EntityMemory completada')
print('   Observa cómo el modelo mantiene un perfil separado de Laura y Marco.')

### 11B — ConversationKGMemory (Knowledge Graph)

KGMemory va un paso más allá: no solo identifica entidades, sino también las **relaciones** entre ellas, formando un grafo de conocimiento.

```
                    trabaja_en
        Laura ─────────────────► Google
          │                         
          │ trabaja_con            
          ▼                         
        Marco ──────────────────► IA Médica
                  desarrolla
```

In [ ]:
# ============================================================
# CELDA 28 — [AVANZADO] Configurar ConversationKGMemory
# ============================================================

from langchain.memory import ConversationKGMemory   # Memoria de grafo de conocimiento

# KGMemory extrae tripletas (sujeto, predicado, objeto) de la conversación.
# Ejemplo: ("Laura", "trabaja_en", "Google")
# El LLM se encarga de identificar estas relaciones en cada turno.
memoria_kg = ConversationKGMemory(
    llm=llm_avanzado,          # LLM para extraer relaciones del texto
    return_messages=True
)

cadena_kg = ConversationChain(
    llm=llm_avanzado,
    memory=memoria_kg,
    verbose=False
)

print('✅ ConversationKGMemory configurada')

In [ ]:
# ============================================================
# CELDA 29 — [AVANZADO] Prueba de KGMemory
# ============================================================

def chatear_kg(mensaje):
    print(f'👤 Usuario: {mensaje}')
    respuesta = cadena_kg.predict(input=mensaje)
    print(f'🤖 Modelo:  {respuesta}')
    print()

    # memoria_kg.kg contiene el grafo de conocimiento.
    # .get_triples() devuelve todas las relaciones extraídas como tripletas.
    try:
        tripletas = memoria_kg.kg.get_triples()
        print(f'🕸️  Grafo de conocimiento ({len(tripletas)} relaciones):')
        for sujeto, predicado, objeto in tripletas:
            print(f'   {sujeto} ──[{predicado}]──► {objeto}')
    except Exception:
        print('   (El grafo aún no tiene relaciones — continúa chateando)')
    print('-' * 60)

chatear_kg('Carlos es físico y trabaja en la Universidad Nacional.')
chatear_kg('Carlos colabora con Elena, quien es matemática.')
chatear_kg('Elena dirige el laboratorio de computación cuántica.')

print('✅ Prueba de KGMemory completada')
print('   Observa las relaciones extraídas: (sujeto) → [predicado] → (objeto)')

---
## PARTE 12 — Publicación con Ngrok

Ngrok crea un túnel seguro desde internet hasta tu servidor local en Colab, generando una URL pública que puedes compartir con toda la clase en tiempo real.

```
  Internet
     │
     ▼  URL pública (https://xxxx.ngrok-free.app)
  Servidores Ngrok
     │
     ▼  túnel cifrado
  Tu sesión de Colab (localhost:7860)
     │
     ▼
  Gradio + LangChain + Ollama
```

In [ ]:
# ============================================================
# CELDA 30 — Instalar pyngrok
# ============================================================

# pyngrok es la librería Python oficial para controlar Ngrok.
# Nos permite autenticar, crear túneles y obtener URLs desde código.
!pip install -q pyngrok

print('✅ pyngrok instalado')

In [ ]:
# ============================================================
# CELDA 31 — Autenticación con el token de Ngrok
# ============================================================

from pyngrok import ngrok          # Librería para controlar Ngrok desde Python
from google.colab import userdata  # Para leer secretos de Colab de forma segura

# Leemos el token desde los Secrets de Colab (ícono de llave en el panel izquierdo).
# Esto es más seguro que poner el token directamente en el código,
# porque el notebook puede compartirse sin exponer credenciales.
try:
    token_ngrok = userdata.get('NGROK_TOKEN')
    print('✅ Token leído desde Colab Secrets')
except Exception:
    # Si no hay token en Secrets, permitimos ingresarlo manualmente.
    # ADVERTENCIA: No compartas el notebook con el token visible aquí.
    print('⚠️  No se encontró NGROK_TOKEN en Secrets.')
    print('   Ve a: Panel izquierdo → 🔑 Secrets → Añade NGROK_TOKEN')
    print('   Obtén tu token gratis en: https://dashboard.ngrok.com')
    token_ngrok = input('   O ingresa el token manualmente (no lo compartas): ')

# Configuramos el token en pyngrok para autenticar todas las sesiones futuras
ngrok.set_auth_token(token_ngrok)
print('✅ Autenticación con Ngrok configurada')

In [ ]:
# ============================================================
# CELDA 32 — Crear el túnel y lanzar la interfaz pública
# ============================================================

# Cerramos cualquier túnel previo para evitar conflictos.
# Si no hay túneles activos, ngrok.kill() no hace nada.
ngrok.kill()

# Cerramos la interfaz Gradio si ya estaba corriendo en el puerto 7860.
# Si no estaba corriendo, demo.close() no hace nada.
try:
    demo.close()
except Exception:
    pass

# Creamos el túnel Ngrok apuntando al puerto 7860 (donde corre Gradio).
# ngrok.connect() devuelve un objeto con la URL pública.
tunel = ngrok.connect(7860)
print(f'🌐 URL pública: {tunel.public_url}')
print('   Comparte este enlace con tus compañeros')

# Relanzamos Gradio.
# server_name='0.0.0.0' es esencial para que Ngrok pueda acceder al servidor.
# share=False porque Ngrok es nuestra solución de compartido.
demo.launch(
    server_name='0.0.0.0',
    server_port=7860,
    share=False,
    quiet=True             # Suprime el output de Gradio para que se vea solo la URL
)

print('✅ Interfaz publicada y accesible desde internet')

In [ ]:
# ============================================================
# CELDA 33 — Verificar túneles activos
# ============================================================

# ngrok.get_tunnels() devuelve la lista de todos los túneles activos.
# Es útil para verificar que el túnel está funcionando correctamente.
tuneles = ngrok.get_tunnels()

print(f'Túneles activos: {len(tuneles)}')
for t in tuneles:
    print(f'  • Público: {t.public_url}')
    print(f'    Local:   {t.config["addr"]}')

print('✅ Verificación de túneles completada')

---
## Celda de Limpieza de Recursos

Ejecuta esta celda al terminar la actividad para liberar todos los recursos del sistema.

In [ ]:
# ============================================================
# CELDA 34 — Limpieza de todos los recursos
# ============================================================

# PASO 1: Cerrar el túnel Ngrok
# Esto libera la URL pública y cierra la conexión con los servidores de Ngrok.
try:
    ngrok.kill()
    print('✅ Túnel Ngrok cerrado')
except Exception as e:
    print(f'   Ngrok ya estaba cerrado: {e}')

# PASO 2: Cerrar la interfaz Gradio
# Libera el puerto 7860 para que pueda usarse en otro notebook.
try:
    demo.close()
    print('✅ Interfaz Gradio cerrada')
except Exception as e:
    print(f'   Gradio ya estaba cerrado: {e}')

# PASO 3: Detener el servidor Ollama
# ollama_process es el objeto Popen que lanzamos en la Celda 3.
try:
    ollama_process.terminate()   # Envía señal SIGTERM para cierre ordenado
    ollama_process.wait()        # Espera a que el proceso termine completamente
    print('✅ Servidor Ollama detenido')
except Exception as e:
    print(f'   Ollama ya estaba detenido: {e}')

# PASO 4: Liberar memoria (si se usó GPU para el módulo avanzado)
# Esto es especialmente importante si usaste llama3.2 que consume más VRAM.
try:
    import torch
    del llm_avanzado             # Elimina el objeto del modelo avanzado
    torch.cuda.empty_cache()     # Libera la VRAM de la GPU
    print('✅ Memoria GPU liberada')
except Exception:
    print('   (No se usó GPU o ya estaba liberada)')

print()
print('✅ Limpieza completa — todos los recursos liberados')

---
## 📊 Rúbrica de Evaluación

| Criterio | Excelente (5) | Satisfactorio (3) | En desarrollo (1) |
|---|---|---|---|
| **Ejecución del notebook** | Todas las celdas ejecutadas sin errores, de arriba a abajo | La mayoría de celdas funcionan; algún error menor no bloqueante | Más de 2 partes no ejecutan correctamente |
| **Comprensión de los tipos de memoria** | Las 5 preguntas de reflexión muestran comprensión clara y diferenciada de cada tipo | Responde correctamente al menos 3 preguntas | Respuestas superficiales o confunde los tipos entre sí |
| **Experimentación con la interfaz** | Demuestra haber probado todos los tipos en Gradio y describe diferencias observadas | Probó al menos 2 tipos y describe una diferencia | Solo usó el tipo por defecto sin explorar |
| **Módulo avanzado** | Ejecutó EntityMemory y KGMemory, observó y describió las entidades/relaciones detectadas | Intentó el módulo avanzado aunque con errores parciales | No intentó el módulo avanzado |
| **Publicación con Ngrok** | URL pública generada, funcional y compartida con el docente | URL generada pero con acceso intermitente | No se pudo generar la URL pública |
| **Retos opcionales** | Completó al menos 1 reto con código funcional y documentado | Intentó un reto con código parcial | No intentó los retos opcionales |